# Visualize Time Profiles: REDS Recall
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import defaultdict
from itertools import chain, combinations, product
import re
import textwrap

import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import seaborn as sns

from hk1_r12ter_analysis.config import (
    GENE_ID,
    PROCESSED_DATA_DIR,
    PROJ_ROOT,
    PROTEIN_ID,
    RAW_DATA_DIR,
    REPORTS_DIR,
    logger,
)
from hk1_r12ter_analysis.io import load_dataframe_from_file, make_filename
from hk1_r12ter_analysis.linkage_disequilibrium import identify_independent_snps
from hk1_r12ter_analysis.util import ceil_to_decimal
from hk1_r12ter_analysis.visualize import (
    plot_correlations_vs_pvalues,
    plot_hemolysis_pvalues_for_snp,
)

## REDS Recall

In [ ]:
sample_key = "RECALL DONOR ID"
group_key = "Day"
source_type = "REDS-Recall"
data_type = "Metabolomics"
norm_str = "median-none-auto"

filetype = "pdf"

input_dirpath = PROCESSED_DATA_DIR / "REDS" / "Normalized" / norm_str
output_dirpath = REPORTS_DIR / "REDS" / "TimeProfiles"
output_dirpath.mkdir(parents=True, exist_ok=True)

### Load data

In [ ]:
input_filename = make_filename(source_type, "Donor", "RBC", "All_Data")
index_col = [key for key in [sample_key, group_key] if key]
df_all_data = load_dataframe_from_file(
    input_dirpath / input_filename,
    index_col=index_col,
    header=[0, 1],
)
df_all_data = df_all_data.convert_dtypes()
df_all_data

### Load genomics metadata

In [ ]:
input_dirpath = PROCESSED_DATA_DIR / "REDS"
input_filename = make_filename("REDS", "Genomics", GENE_ID, "Metadata")
df_genomics_ann = load_dataframe_from_file(input_dirpath / input_filename, index_col="ID")
df_genomics_ann = df_genomics_ann.loc[df_all_data.loc[:, f"Genomics-{GENE_ID}"].columns.tolist()]
df_genomics_ann = df_genomics_ann.drop("ANN_Priority", axis=1)
df_rsid_map = df_genomics_ann["RSID"].replace(":", ".")
df_genomics_ann

### Plot abundances over time with confidence intervals

In [ ]:
rsid = "i_rs10762276:71052469:C:T"
# rsid = "i_rs7904932:71150946:C:T"
time = "Day"

zorder = 4
xmajor_tick_multiple = 10
ymajor_tick_multiple = 0.5
palette = {0.0: "xkcd:pale blue", 1.0: "xkcd:light blue", 2.0: "xkcd:lightish blue"}

legend_labels = {0: "HMOZ Ref.", 1: "HTRZ", 2: "HMOZ Alt."}
title_fontdict = {"size": "x-large", "weight": "bold"}
label_fontdict = {"size": "large"}
tick_fontdict = {"size": "medium"}

xlim = df_all_data.index.get_level_values(time).unique()
xlim = (min(xlim), max(xlim))
xlim

#### Glycolysis

In [ ]:
metabolites = [
    "Phosphate",
    "ATP",
    "ADP",
    "AMP",
    "D-Glucose",
    "D-Hexose-phosphate",
    "D-Fructose 1-6-bisphosphate",
    "D-Glyceraldehyde 3-phosphate/Glycerone phosphate",
    "NAD+",
    "2-3-Bisphosphoglycerate",
    "2/3-Phospho-D-glycerate",
    "Phosphoenolpyruvate",
    "Pyruvate",
    "Lactate",
]
data = df_all_data.loc[
    :, [(f"Genomics-{GENE_ID}", rsid)] + [(data_type, met) for met in metabolites]
].dropna()
data = data.droplevel(0, axis=1).reset_index(drop=False)
for metabolite in metabolites:
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 4))
    ax = sns.lineplot(
        data=data,
        x=time,
        y=metabolite,
        hue=rsid,
        # marker="o",
        # markersize=10,
        # markeredgecolor="black",
        palette=palette,
        zorder=zorder,
        linewidth=4,
        estimator="mean",
        errorbar=("ci", 95),
        n_boot=10000,
        seed=4,
        orient="x",
        sort=True,
        err_style="band",
        err_kws=dict(),
        legend=True,
        ax=ax,
        clip_on=False,
    )
    ax = sns.lineplot(
        data=data,
        x=time,
        y=metabolite,
        hue=rsid,
        marker="o",
        markersize=10,
        markeredgecolor="black",
        linestyle="None",
        palette=palette,
        zorder=zorder + 1,
        linewidth=4,
        estimator="mean",
        # errorbar=('ci', 95),
        n_boot=10000,
        seed=4,
        orient="x",
        sort=True,
        err_style="band",
        err_kws=dict(),
        legend=False,
        ax=ax,
        clip_on=False,
    )
    for collection in ax.findobj(mpl.collections.FillBetweenPolyCollection):
        collection.set_color("xkcd:grey")
        collection.set_zorder(2)

    ax.hlines(y=0, xmin=xlim[0], xmax=xlim[-1], linestyle="--", color="black", zorder=zorder)
    ax.set_title(metabolite, fontdict=title_fontdict)
    ax.set_xlim(xlim)
    ax.set_xlabel(f"Storage Duration {time}", fontdict=label_fontdict)
    ax.xaxis.set_major_locator(mpl.ticker.MultipleLocator(xmajor_tick_multiple))
    ax.xaxis.set_minor_locator(
        mpl.ticker.MultipleLocator(xmajor_tick_multiple, offset=xmajor_tick_multiple / 2)
    )
    ax.xaxis.set_tick_params(labelsize=tick_fontdict["size"])

    # ax.set_ylim((-0.8, 2))
    ax.set_ylabel("Normalized Abundance", fontdict=label_fontdict)
    ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(ymajor_tick_multiple))
    ax.yaxis.set_minor_locator(
        mpl.ticker.MultipleLocator(ymajor_tick_multiple, offset=ymajor_tick_multiple / 2)
    )
    ax.yaxis.set_tick_params(labelsize=tick_fontdict["size"])
    sns.despine(ax=ax)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles, labels=[legend_labels[int(x)] for x in labels])

    filename_args = [
        source_type,
        metabolite.replace("/", "-or-").replace(":", "_").replace(",", "_"),
    ]
    output_filename = make_filename(*filename_args, filetype=filetype)
    output_dirpath_rsid = output_dirpath / df_rsid_map.get(rsid, rsid.replace(":", "."))
    output_dirpath_rsid.mkdir(exist_ok=True, parents=True)
    fig.savefig(output_dirpath_rsid / output_filename)
    print(output_filename)
    plt.show()
    plt.close()

#### Pentose phosphate pathway

In [ ]:
metabolites = [
    "6-Phospho-D-gluconate",
    "Sedoheptulose 7-phosphate",
    "Pentose phosphates",
    "D-Erythrose 4-phosphate",
    "NADP+",
]

zorder = 4
xmajor_tick_multiple = 10
ymajor_tick_multiple = 0.5
palette = {0.0: "xkcd:light blue", 1.0: "xkcd:lightish blue", 2.0: "xkcd:true blue"}
palette = {0.0: "xkcd:pale blue", 1.0: "xkcd:light blue", 2.0: "xkcd:lightish blue"}

legend_labels = {0: "HMOZ Ref.", 1: "HTRZ", 2: "HMOZ Alt."}
title_fontdict = {"size": "x-large", "weight": "bold"}
label_fontdict = {"size": "large"}
tick_fontdict = {"size": "medium"}

data = df_all_data.loc[
    :, [(f"Genomics-{GENE_ID}", rsid)] + [(data_type, met) for met in metabolites]
].dropna()
data = data.droplevel(0, axis=1).reset_index(drop=False)
xlim = (data[time].min(), data[time].max())

for metabolite in metabolites:
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 4))
    ax = sns.lineplot(
        data=data,
        x=time,
        y=metabolite,
        hue=rsid,
        # marker="o",
        # markersize=10,
        # markeredgecolor="black",
        palette=palette,
        zorder=zorder,
        linewidth=4,
        estimator="mean",
        errorbar=("ci", 95),
        n_boot=10000,
        seed=4,
        orient="x",
        sort=True,
        err_style="band",
        err_kws=dict(),
        legend=True,
        ax=ax,
        clip_on=False,
    )
    ax = sns.lineplot(
        data=data,
        x=time,
        y=metabolite,
        hue=rsid,
        marker="o",
        markersize=10,
        markeredgecolor="black",
        linestyle="None",
        palette=palette,
        zorder=zorder + 1,
        linewidth=4,
        estimator="mean",
        # errorbar=('ci', 95),
        n_boot=10000,
        seed=4,
        orient="x",
        sort=True,
        err_style="band",
        err_kws=dict(),
        legend=False,
        ax=ax,
        clip_on=False,
    )
    for collection in ax.findobj(mpl.collections.FillBetweenPolyCollection):
        collection.set_color("xkcd:grey")
        collection.set_zorder(2)

    ax.hlines(y=0, xmin=xlim[0], xmax=xlim[-1], linestyle="--", color="black", zorder=zorder)
    ax.set_title(metabolite, fontdict=title_fontdict)
    ax.set_xlim(xlim)
    ax.set_xlabel(f"Storage Duration {time}", fontdict=label_fontdict)
    ax.xaxis.set_major_locator(mpl.ticker.MultipleLocator(xmajor_tick_multiple))
    ax.xaxis.set_minor_locator(
        mpl.ticker.MultipleLocator(xmajor_tick_multiple, offset=xmajor_tick_multiple / 2)
    )
    ax.xaxis.set_tick_params(labelsize=tick_fontdict["size"])

    # ax.set_ylim((-0.8, 2))
    ax.set_ylabel("Normalized Abundance", fontdict=label_fontdict)
    ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(ymajor_tick_multiple))
    ax.yaxis.set_minor_locator(
        mpl.ticker.MultipleLocator(ymajor_tick_multiple, offset=ymajor_tick_multiple / 2)
    )
    ax.yaxis.set_tick_params(labelsize=tick_fontdict["size"])
    sns.despine(ax=ax)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles, labels=[legend_labels[int(x)] for x in labels])

    filename_args = [source_type, metabolite.replace("-", "_").replace(" ", "_")]
    output_filename = make_filename(*filename_args, filetype=filetype)
    output_dirpath_rsid = output_dirpath / df_rsid_map.get(rsid, rsid.replace(":", "."))
    output_dirpath_rsid.mkdir(exist_ok=True, parents=True)
    fig.savefig(output_dirpath_rsid / output_filename)
    plt.show()
    plt.close()